# Git Live-Demo: `fetch`, `pull` und Merge-Konflikte

Dieses Notebook ist als **Trainingsleitfaden für eine Live-Session** gedacht.  
Es ist bewusst für **Windows / PowerShell** aufgebaut und zeigt Schritt für Schritt:

- den Unterschied zwischen `git fetch` und `git pull`
- den Unterschied zwischen **lokalem Branch** und **Remote-Tracking-Branch**
- wie ein Merge-Konflikt entsteht
- wie man ihn sauber auflöst

---

## Lernziele

Am Ende sollten die Teilnehmer verstanden haben:

1. `git fetch` holt neue Informationen vom Remote, **ohne** den lokalen Arbeitsstand direkt zu verändern.
2. `git pull` ist praktisch **`fetch` + Integration** in den aktuellen Branch.
3. Ein sichtbarer Remote-Branch wie `origin/main` ist **nicht automatisch** ein lokaler Branch `main`.
4. Merge-Konflikte entstehen dann, wenn Git zwei Änderungen **nicht fachlich eindeutig automatisch zusammenführen kann**.

## Didaktischer Ablauf

Die Session ist in vier Blöcke aufgeteilt:

### Block 1 – Setup
Wir erzeugen ein kleines zentrales Remote-Repository und zwei lokale Arbeitskopien.

### Block 2 – `fetch` vs `pull`
Wir zeigen erst einen sauberen, konfliktfreien Fall.

### Block 3 – Lokaler Branch vs. Remote-Branch
Wir greifen den Fall auf, dass nach dem Klonen nur `origin/main` sichtbar sein kann.

### Block 4 – Merge-Konflikt
Wir ändern bewusst dieselbe Zeile in zwei Arbeitskopien unterschiedlich und lösen den Konflikt gemeinsam.

---

## Rollenbild in der Demo

Wir simulieren zwei Entwickler:

- **Entwickler A** → Ordner `dev-a`
- **Entwickler B** → Ordner `dev-b`

Dazu gibt es ein zentrales Remote-Repository:

- `git-demo-remote.git`

Als gemeinsame Datei verwenden wir:

- `welcome.txt`

Die Datei ist bewusst einfach gehalten, damit jede Änderung sofort sichtbar ist.

## Wichtiger Hinweis zu JupyterLab

Die folgenden Befehle sind **PowerShell-Befehle**.  
Wenn du dieses Notebook in JupyterLab ausführst, gibt es zwei sinnvolle Varianten:

- Du nutzt ein **PowerShell-Kernel**, dann kannst du die Befehle direkt ausführen.
- Oder du nutzt einen Python-Kernel und behandelst das Notebook als **Moderations- und Demo-Skript**, während du die Befehle parallel in einem PowerShell-Terminal ausführst.

Eine **separate Anleitung zur Einrichtung eines PowerShell-Kernels in VS Code** findest du hier: [separate Anleitung zur Einrichtung eines PowerShell-Kernels in VS Code](./powershell_kernel_in_vscode_anleitung.ipynb)

Für eine Live-Schulung ist die zweite Variante oft angenehmer, weil du die Ausgaben sauber im Terminal zeigen kannst.


# Block 1 – Setup

## 1. Demo-Ordner anlegen

Wir erzeugen zunächst einen gemeinsamen Demo-Ordner.

In [ ]:

mkdir git-demo
cd git-demo

## 2. Zentrales Bare-Repository erstellen

Dieses Repository übernimmt die Rolle des „Servers“ bzw. des zentralen Remote-Repositories.

In [ ]:

git init --bare git-demo-remote.git

## 3. Erste Arbeitskopie klonen: `dev-a`

Jetzt klonen wir das zentrale Repository in eine erste Arbeitskopie für **Entwickler A**.

In [ ]:

git clone .\git-demo-remote.git dev-a
cd .\dev-a

## 4. Lokalen Branch in `dev-a` sauber auf `main` setzen

Das sorgt für eine stabile Ausgangslage und vermeidet Branch-Verwirrung.

In [ ]:

git switch -c main

## 5. Beispiel-Datei anlegen

Wir erzeugen eine kleine Textdatei, mit der wir die Änderungen später gut nachvollziehen können.

In [ ]:

"Willkommen zum Git-Workshop" | Out-File -Encoding utf8 welcome.txt
"Version 1" | Add-Content welcome.txt

## 6. Ersten Commit erzeugen

In [ ]:

git add welcome.txt
git commit -m "Initial commit"

## 7. Den Startzustand auf das Remote pushen

Das `-u` sorgt dafür, dass der lokale Branch `main` mit `origin/main` verknüpft wird.

In [ ]:

git push -u origin main

## 8. Zweite Arbeitskopie klonen: `dev-b`

Jetzt klonen wir das gleiche Remote-Repository ein zweites Mal für **Entwickler B**.

In [ ]:

cd ..
git clone .\git-demo-remote.git dev-b
cd .\dev-b

# Block 2 – Lokaler Branch vs. Remote-Branch

## 9. Branches prüfen

Hier tritt oft ein didaktisch sehr spannender Fall auf:  
Es kann sein, dass nach dem Klonen zunächst **nur** der Remote-Tracking-Branch sichtbar ist.

In [ ]:

git branch -a

### Typischer Fall

Es kann zum Beispiel nur Folgendes erscheinen:

```text
remotes/origin/main
```

Das bedeutet:

- Das Remote kennt einen Branch `main`
- Lokal existiert aber **noch kein eigener Arbeitsbranch `main`**

Genau das ist ein wichtiger Lernpunkt:  
**`origin/main` ist nicht automatisch dasselbe wie ein lokaler Branch `main`.**

## 10. Lokalen Branch `main` aus `origin/main` erzeugen

Jetzt erzeugen wir bewusst einen lokalen Branch und hängen ihn an den Remote-Branch an. 

Wir machen das, weil origin/main zunächst nur der Remote-Tracking-Branch ist. Er zeigt uns also nur, wie der Stand auf dem Remote aussieht, ist aber noch kein eigener lokaler Arbeitsbranch, auf dem wir direkt normal arbeiten, committen oder Vergleiche wie git diff main origin/main sinnvoll ausführen können.

Mit diesem Schritt erstellen wir deshalb einen lokalen Branch main und verknüpfen ihn gleichzeitig mit origin/main. Dadurch haben wir einen echten Arbeitsbranch, auf dem wir uns bewegen, während Git gleichzeitig weiß, zu welchem Remote-Branch dieser gehört. Erst damit ist die typische Arbeitsweise mit pull, push, status und sauberen Vergleichen wirklich klar und konsistent.

In [ ]:

git switch -c main --track origin/main

## 11. Kontrolle der Branch-Situation

Jetzt sollte sowohl der lokale Branch als auch der Remote-Tracking-Branch sichtbar sein. Prüfen können wir das mit:

In [ ]:

git branch -a

### Erwartete Ausgabe

Typischerweise sieht das dann so aus:

```text
* main
  remotes/origin/main
```

Hier kannst du den Teilnehmern erklären:

- `main` = mein lokaler Arbeitsbranch
- `origin/main` = der Stand auf dem Remote, dem mein lokaler Branch folgt

# Block 3 – `git fetch` vs. `git pull`

## 12. In `dev-a` eine Änderung vornehmen und pushen

Jetzt simulieren wir, dass **Entwickler A** weiterarbeitet und eine Änderung auf den Server schiebt.

In [ ]:

cd ..\dev-a
"Neue Zeile von Entwickler A" | Add-Content welcome.txt
git add welcome.txt
git commit -m "Ergänzung durch Entwickler A"
git push

## 13. In `dev-b` den aktuellen Stand prüfen

Bevor wir irgendetwas holen, schauen wir zunächst nach dem Status und dem bisherigen Verlauf.

In [ ]:

cd ..\dev-b
git status
git log --oneline --all --graph

## `git log --oneline --all --graph`

Dieser Befehl zeigt den Commit-Verlauf kompakt und grafisch an.

### Bedeutung der einzelnen Teile

- `git log`  
  Zeigt grundsätzlich die Historie aller Commits.

- `--oneline`  
  Zeigt jeden Commit nur in einer Zeile an:  
  Commit-ID + Commit-Nachricht.

- `--all`  
  Zeigt nicht nur den aktuellen Branch, sondern alle Branches und Remote-Branches.

- `--graph`  
  Zeichnet eine kleine ASCII-Grafik davor, damit man Verzweigungen und Merges erkennen kann.

### Mögliche Ausgabe:

```text
* a1b2c3d Version durch Entwickler B geändert
| * e4f5g6h Version durch Entwickler A geändert
|/
* 7h8i9j0 Ergänzung durch Entwickler A
* 1a2b3c4 Initial commit
```

### Erklärung der Ausgabe

- `*` steht jeweils für einen Commit.
- Die Linien `|`, `/` und `\` zeigen, wie sich Branches voneinander getrennt oder später wieder zusammengeführt haben.
- Unterschiedlich eingerückte Linien bedeuten, dass mehrere Entwicklungsstände parallel existieren.
- Wenn zwei Linien später wieder zusammenlaufen, wurde ein Merge durchgeführt.

Im Beispiel haben Entwickler A und Entwickler B jeweils unabhängig voneinander einen eigenen Commit erstellt. Git zeigt deshalb zwei parallele Linien im Verlauf. Dadurch wird sichtbar, dass beide Änderungen gleichzeitig existieren und eventuell noch zusammengeführt werden müssen.

Gerade bei `fetch`, `pull` und Merge-Konflikten ist dieser Befehl sehr hilfreich, weil man sofort erkennen kann:

- Wo mein aktueller lokaler Branch steht
- Wo `origin/main` steht
- Ob es neue Commits auf dem Remote gibt
- Ob mein lokaler Stand hinter dem Remote liegt
- Ob bereits ein Merge durchgeführt wurde
- Ob zwei Entwickler parallel gearbeitet haben

## 14. `git fetch` ausführen

Jetzt holen wir die neuen Informationen vom Remote, **ohne** unseren lokalen Arbeitsstand sofort zu verändern.

In [ ]:

git fetch

## 15. Nach `fetch` erneut prüfen

Jetzt sollte Git den neuen Stand auf dem Remote kennen.  
Die Arbeitsdatei selbst ist aber noch nicht automatisch aktualisiert.

In [ ]:

git log --oneline --all --graph
git diff HEAD origin/main
Get-Content .\welcome.txt

### Knackpunkt

Hier kann man schön sehen:

- `fetch` lädt nur Informationen
- der lokale Branch wurde noch nicht automatisch integriert
- die Datei `welcome.txt` ist noch auf dem alten lokalen Stand

Oder kurz und knapp:

- `fetch` = "erst mal nachschauen"
- `pull` = "holen und direkt zusammenführen"

## 16. Jetzt `git pull`

Nun integrieren wir die Änderungen vom Remote in den lokalen Branch.

In [ ]:

git pull
Get-Content .\welcome.txt

### Beobachtung

Nach `pull` ist die neue Zeile von Entwickler A jetzt auch im lokalen Arbeitsstand von `dev-b` angekommen.

Hier kann man sehr schön zusammenfassen:

- `fetch` ändert zunächst nur dein Wissen über das Remote
- `pull` ändert zusätzlich deinen lokalen Branch

# Block 4 – Merge-Konflikt absichtlich erzeugen

## Ausgangssituation

Die Datei enthält aktuell ungefähr diesen Inhalt:

```text
Willkommen zum Git-Workshop
Version 1
Neue Zeile von Entwickler A
```

Jetzt werden **beide Entwickler dieselbe Zeile unterschiedlich ändern**.  
Genau dadurch erzwingen wir einen Konflikt.

## 17. In `dev-a` dieselbe Zeile ändern und pushen

Wir ändern in `dev-a` die Zeile `Version 1` in eine neue Variante.

In [ ]:

cd ..\dev-a
notepad .\welcome.txt

### Dateiinhalt in `dev-a`

Ändere die Datei auf:

```text
Willkommen zum Git-Workshop
Version 2 von Entwickler A
Neue Zeile von Entwickler A
```

Dann speichern und committen:

In [ ]:

git add welcome.txt
git commit -m "Version durch Entwickler A geändert"
git push

## 18. In `dev-b` dieselbe Zeile anders ändern

Jetzt ändern wir **dieselbe Zeile** in `dev-b`, aber mit anderem Inhalt.

In [ ]:

cd ..\dev-b
notepad .\welcome.txt

### Dateiinhalt in `dev-b`

Ändere die Datei auf:

```text
Willkommen zum Git-Workshop
Version 2 von Entwickler B
Neue Zeile von Entwickler A
```

Dann speichern und lokal committen, **aber noch nicht pushen**.

In [ ]:

git add welcome.txt
git commit -m "Version durch Entwickler B geändert"

## 19. Jetzt den Konflikt auslösen

Wenn wir nun in `dev-b` ein `git pull` ausführen, muss Git versuchen, beide Änderungen zusammenzuführen.

In [ ]:

git pull

### Erwartete Meldung

Typischerweise erscheint jetzt eine Konfliktmeldung wie:

```text
Auto-merging welcome.txt
CONFLICT (content): Merge conflict in welcome.txt
Automatic merge failed; fix conflicts and then commit the result.
```

Das ist der eigentliche Aha-Moment der Demo.

## 20. Konfliktdatei anzeigen

Jetzt schauen wir in die Datei hinein.

In [ ]:

Get-Content .\welcome.txt

### Typische Konfliktmarker

Der Inhalt sieht nun ungefähr so aus:

```text
Willkommen zum Git-Workshop
<<<<<<< HEAD
Version 2 von Entwickler B
=======
Version 2 von Entwickler A
>>>>>>> abc1234
Neue Zeile von Entwickler A
```

Das heißt jetzt ungefähr das:

- `<<<<<<< HEAD` = unsere lokale Version
- `=======` = Trenner
- `>>>>>>> ...` = Version aus dem anderen Commit / vom Remote

Git sagt also nicht "ich bin kaputt", sondern:  
**"Hier gibt es zwei konkurrierende Änderungen. Bitte entscheide du."**

## 21. Konflikt manuell auflösen

Jetzt bereinigen wir die Datei gemeinsam.

In [ ]:

notepad .\welcome.txt

### Beispiel für eine aufgelöste Fassung

Zum Beispiel so:

```text
Willkommen zum Git-Workshop
Version 2 final abgestimmt von A und B
Neue Zeile von Entwickler A
```

Danach speichern.

## 22. Auflösung committen

In [ ]:

git add welcome.txt
git commit -m "Merge-Konflikt in welcome.txt aufgelöst"

## 23. Verlauf anzeigen

Zum Abschluss schauen wir uns die Commit-Historie an.

In [ ]:

git log --oneline --all --graph

# Typische Stolperfallen

## 1. Branch-Namen blind annehmen
Nicht jedes Repo nutzt automatisch `main`. Manche nutzen `master`, manche haben lokal zunächst nur den Remote-Tracking-Branch.

Darum ist diese Prüfung immer sinnvoll:

```powershell
git branch -a
```

## 2. Remote-Branch mit lokalem Branch verwechseln
`origin/main` ist nicht automatisch dein lokaler Arbeitsbranch `main`.

## 3. `pull` als Reflex benutzen
Viele nutzen immer direkt `git pull`, ohne vorher zu prüfen, was sich verändert hat.  Das sollte unbedingt vermieden werden.

## 4. Konflikte als "Fehler von Git" missverstehen
Ein Konflikt bedeutet meist nur, dass zwei Änderungen fachlich kollidieren und Git keine sichere Entscheidung treffen will.

# Kompaktübersicht aller Befehle

## Setup

```powershell
mkdir git-demo
cd git-demo
git init --bare git-demo-remote.git
git clone .\git-demo-remote.git dev-a
cd .\dev-a
git switch -c main
"Willkommen zum Git-Workshop" | Out-File -Encoding utf8 welcome.txt
"Version 1" | Add-Content welcome.txt
git add welcome.txt
git commit -m "Initial commit"
git push -u origin main
cd ..
git clone .\git-demo-remote.git dev-b
cd .\dev-b
git branch -a
git switch -c main --track origin/main
git branch -a
```

## Fetch vs Pull

```powershell
cd ..\dev-a
"Neue Zeile von Entwickler A" | Add-Content welcome.txt
git add welcome.txt
git commit -m "Ergänzung durch Entwickler A"
git push

cd ..\dev-b
git status
git log --oneline --all --graph
git fetch
git log --oneline --all --graph
git diff HEAD origin/main
Get-Content .\welcome.txt
git pull
Get-Content .\welcome.txt
```

## Konflikt erzeugen

```powershell
cd ..\dev-a
notepad .\welcome.txt
git add welcome.txt
git commit -m "Version durch Entwickler A geändert"
git push

cd ..\dev-b
notepad .\welcome.txt
git add welcome.txt
git commit -m "Version durch Entwickler B geändert"
git pull
```

## Konflikt lösen

```powershell
Get-Content .\welcome.txt
notepad .\welcome.txt
git add welcome.txt
git commit -m "Merge-Konflikt in welcome.txt aufgelöst"
git log --oneline --all --graph
```

# Abschlussgedanke für die Session

Die eigentliche Stärke dieser Demo ist nicht nur, dass die Teilnehmer zwei oder drei Git-Befehle sehen.  
Der eigentliche Mehrwert ist, dass sie verstehen, **was Git intern gerade tut**.

Wenn dieser Unterschied sitzt, werden spätere Probleme deutlich weniger mystisch:

- `fetch` holt Informationen
- `pull` integriert
- Konflikte sind kein Chaos, sondern ein Hinweis auf widersprüchliche Änderungen
- lokale Branches und Remote-Branches sind verwandt, aber nicht identisch

Genau dann beginnt Git für die Teilnehmer meistens zum ersten Mal wirklich Sinn zu ergeben.